# Notebook 04 — Model Training
Credit Risk Scoring & Default Prediction

## Objetivo
Comparar cinco modelos mediante validación cruzada estratificada exclusivamente sobre entrenamiento. Seleccionar un candidato por average precision (AP) y guardar su pipeline completo. La prueba final permanece cerrada hasta el Notebook 05. Fuente: [UCI 350](https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients), vía Notebook 03. Se predice incumplimiento del siguiente mes, no PD anual. Este dataset histórico no representa una validación externa para banca mexicana.

## 1. Configuración
Si faltan paquetes, ejecuta en una celda `%pip install scikit-learn xgboost lightgbm joblib pandas numpy scipy matplotlib` y reinicia el kernel. Si sklearn vuelve a fallar, selecciona Python (Credit Risk) o repara el entorno antes de continuar. Usaremos cinco folds, semilla 42 y configuraciones iniciales, sin optimización de hiperparámetros.

In [5]:
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import joblib
from scipy.stats import ks_2samp
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import make_scorer
from IPython.display import display

RANDOM_STATE = 42
TARGET = "DEFAULT"
root = Path.cwd()
if root.name.lower() == "notebooks":
    root = root.parent
processed = root / "Data" / "processed"
models_dir = root / "models"
figures = root / "reports" / "figures"
print(sys.executable, sklearn.__version__)
print(root)

C:\Users\pirom\anaconda3\python.exe 1.8.0
C:\Users\pirom\OneDrive\Documentos\Proyectos\credit_risk_scoring


## 2. Cargar exclusivamente el conjunto de entrenamiento
El manifiesto de la Fase 3 fija el orden de predictores. `row_id` identifica registros, no es una variable predictora. Las variables demográficas se excluyen del modelo base y se reservan para auditorías. No se abre el archivo de test.

In [7]:
train_path = processed / "credit_risk_train_features.csv"
manifest_path = processed / "feature_manifest.json"
if not train_path.exists() or not manifest_path.exists():
    raise FileNotFoundError("Ejecuta Notebook 03 y revisa la carpeta: " + str(processed))
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
features = manifest["features"]
audit_only = manifest["audit_only"]
train = pd.read_csv(train_path)
assert manifest["target"] == TARGET
assert len(features) == len(set(features))
assert set(features).isdisjoint([TARGET, "row_id"] + audit_only)
assert set(train.columns) == set(features + [TARGET, "row_id"]), "Esquema incompatible"
assert train["row_id"].is_unique and train["row_id"].notna().all()
assert train[TARGET].isin([0, 1]).all()
X_train = train[features].apply(pd.to_numeric, errors="raise").replace([np.inf,-np.inf],np.nan)
y_train = train[TARGET].astype(int)
assert y_train.value_counts().min() >= 5
assert not X_train.isna().all().any(), "Hay predictores completamente vacíos"
print("Train:", X_train.shape)
print("Tasa de default:", y_train.mean())
display(X_train.isna().sum().sort_values(ascending=False).head(10))

Train: (24000, 34)
Tasa de default: 0.22120833333333334


payment_to_previous_bill    2508
LIMIT_BAL                      0
bill_change                    0
late_months                    0
late_2plus_months              0
max_delay                      0
delay_change                   0
bill_mean                      0
bill_std                       0
payment_mean                   0
dtype: int64

## 3. Definir los modelos y pipelines
Dummy usa la prevalencia como referencia. Regresión logística necesita escalado; los árboles no. La imputación de mediana se ajusta dentro de cada fold, nunca sobre todos los datos antes de validar. No usaremos SMOTE ni ponderación inicialmente para establecer una línea base de probabilidades; evaluaremos alternativas después. XGBoost y LightGBM son opcionales: si no están instalados, se informa y se comparan los demás.

In [8]:
def pipeline(estimator, scale=False):
    steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale:
        steps.append(("scaler", StandardScaler()))
    steps.append(("model", estimator))
    return Pipeline(steps)

candidates = {
    "Dummy": pipeline(DummyClassifier(strategy="prior")),
    "LogisticRegression": pipeline(LogisticRegression(max_iter=2000, random_state=RANDOM_STATE), True),
    "RandomForest": pipeline(RandomForestClassifier(n_estimators=150, min_samples_leaf=10, random_state=RANDOM_STATE, n_jobs=2)),
}
try:
    from xgboost import XGBClassifier
    candidates["XGBoost"] = pipeline(XGBClassifier(n_estimators=150, max_depth=3, learning_rate=.05, subsample=.85, colsample_bytree=.85, eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=2))
except ImportError as e:
    print("XGBoost omitido:", e)
try:
    from lightgbm import LGBMClassifier
    candidates["LightGBM"] = pipeline(LGBMClassifier(n_estimators=150, learning_rate=.05, num_leaves=15, random_state=RANDOM_STATE, n_jobs=2, verbosity=-1))
except ImportError as e:
    print("LightGBM omitido:", e)
print(list(candidates))

['Dummy', 'LogisticRegression', 'RandomForest', 'XGBoost', 'LightGBM']


## 4. Validación cruzada y métricas
AP resume la curva precision-recall (no es idéntica al área trapezoidal PR-AUC). ROC-AUC mide ordenamiento, Gini = 2 AUC − 1 y KS mide la máxima separación entre distribuciones de scores de incumplidores y no incumplidores. Brier mide error cuadrático de probabilidades (menor es mejor). Precision, recall, F1 y accuracy usan el umbral predeterminado de cada clasificador, generalmente 0.5. La elección se realiza por AP media, no por accuracy. Los mismos folds se usan para todos.

In [9]:
def ks_score(estimator, X, y):
    prob = estimator.predict_proba(X)[:, 1]
    y = np.asarray(y)
    return ks_2samp(prob[y == 0], prob[y == 1]).statistic

scoring = {
    "roc_auc": "roc_auc", "ap": "average_precision",
    "accuracy": "accuracy", "precision": make_scorer(__import__('sklearn.metrics', fromlist=['precision_score']).precision_score, zero_division=0),
    "recall": "recall", "f1": "f1", "ks": ks_score,
    "brier": "neg_brier_score"
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
folds = list(cv.split(X_train, y_train))
rows = []
for name, model in candidates.items():
    print("Evaluando:", name, flush=True)
    scores = cross_validate(model, X_train, y_train, cv=folds, scoring=scoring, n_jobs=1, error_score="raise")
    for i in range(len(folds)):
        row = {"model": name, "fold": i+1, "fit_seconds": scores["fit_time"][i]}
        row.update({metric: float(scores["test_"+metric][i]) for metric in scoring})
        row["brier"] *= -1
        row["gini"] = 2 * row["roc_auc"] - 1
        rows.append(row)
fold_results = pd.DataFrame(rows)
metrics = list(scoring) + ["gini"]
summary = fold_results.groupby("model")[metrics].mean().sort_values("ap", ascending=False)
summary["ap_std"] = fold_results.groupby("model")["ap"].std()
display(summary.round(4))

Evaluando: Dummy


Evaluando: LogisticRegression


Evaluando: RandomForest


Evaluando: XGBoost


Evaluando: LightGBM


C:\Users\pirom\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\pirom\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\pirom\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


C:\Users\pirom\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\pirom\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\pirom\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


C:\Users\pirom\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\pirom\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\pirom\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


C:\Users\pirom\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\pirom\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\pirom\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


C:\Users\pirom\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\pirom\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\pirom\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,roc_auc,ap,accuracy,precision,recall,f1,ks,brier,gini,ap_std
model,,,,,,,,,,
XGBoost,0.7868,0.5613,0.8215,0.6766,0.3697,0.4782,0.4405,0.1334,0.5736,0.0091
LightGBM,0.7848,0.5608,0.8219,0.6775,0.3720,0.4803,0.4377,0.1337,0.5697,0.0073
RandomForest,0.7819,0.5555,0.8195,0.6685,0.3658,0.4728,0.4348,0.1344,0.5638,0.0066
LogisticRegression,0.7702,0.5301,0.8169,0.6774,0.3296,0.4433,0.4215,0.1374,0.5403,0.0145
Dummy,0.5000,0.2212,0.7788,0.0000,0.0000,0.0000,0.0000,0.1723,0.0000,0.0001


## 5. Comparación visual
AP debe compararse con la prevalencia de entrenamiento; ROC-AUC con 0.5. Las barras muestran medias de folds, mientras que la desviación estándar de AP describe variabilidad entre folds (no es un intervalo de confianza).

In [10]:
ax = summary["ap"].sort_values().plot.barh(xerr=summary["ap_std"].reindex(summary["ap"].sort_values().index), figsize=(9,4), capsize=3)
ax.axvline(y_train.mean(), linestyle="--", color="gray", label="Prevalencia train")
ax.set(xlabel="Average precision (media CV)", title="Comparación de modelos — entrenamiento")
ax.legend(); plt.tight_layout()
figures.mkdir(parents=True, exist_ok=True)
plt.savefig(figures / "model_cv_average_precision.png", dpi=150, bbox_inches="tight")
plt.show()

<Figure size 900x400 with 1 Axes>

## 6. Seleccionar y ajustar candidato
Se selecciona el mejor modelo no Dummy por AP, con ROC-AUC como desempate. Si no supera Dummy se emite una advertencia: no se considera listo para uso. El candidato se ajusta sobre todo train. La selección de modelo y cualquier optimización posterior puede sobreestimar el resultado CV; evaluaremos en test una vez congeladas las decisiones. La probabilidad aún no está calibrada.

In [11]:
ranked = summary.drop(index="Dummy").sort_values(["ap", "roc_auc"], ascending=False)
best_name = ranked.index[0]
best_model = candidates[best_name]
if ranked.iloc[0].ap <= summary.loc["Dummy", "ap"]:
    print("Advertencia: el candidato no supera la referencia Dummy.")
best_model.fit(X_train, y_train)
prob = best_model.predict_proba(X_train.head(5))[:, 1]
assert np.isfinite(prob).all() and ((prob>=0)&(prob<=1)).all()
print("Candidato:", best_name)
print("AP CV:", ranked.iloc[0].ap)
print("ROC-AUC CV:", ranked.iloc[0].roc_auc)
print("Probabilidades de ejemplo en train:", prob.round(4))

Candidato: XGBoost
AP CV: 0.5612786566814469
ROC-AUC CV: 0.7867971480695545
Probabilidades de ejemplo en train: [0.6891 0.3196 0.3316 0.1195 0.2384]


## 7. Exportar resultados
Guardamos el pipeline completo para que el preprocesamiento se aplique de igual forma en inferencia. Los archivos joblib solo deben cargarse desde fuentes confiables y con versiones compatibles.

In [12]:
models_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(best_model, models_dir / "best_model.joblib")
fold_results.to_csv(processed / "model_cv_folds.csv", index=False)
summary.to_csv(processed / "model_cv_summary.csv")
metadata = {"model": best_name, "target": TARGET, "features": features, "metric_selection": "average_precision", "random_state": RANDOM_STATE, "n_splits": 5, "sklearn_version": sklearn.__version__, "cv_ap": float(ranked.iloc[0].ap), "cv_roc_auc": float(ranked.iloc[0].roc_auc), "models_compared": list(candidates), "test_used": False, "calibrated": False}
(models_dir / "model_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
reloaded = joblib.load(models_dir / "best_model.joblib")
np.testing.assert_allclose(reloaded.predict_proba(X_train.head(5))[:,1], prob)
print("Guardado:", models_dir / "best_model.joblib")

Guardado: C:\Users\pirom\OneDrive\Documentos\Proyectos\credit_risk_scoring\models\best_model.joblib


## 8. Interpretación y siguiente fase
Revisa AP, ROC-AUC, Gini, KS y Brier, junto con la variabilidad entre folds. Un modelo con AP alta puede tener recall bajo al umbral 0.5; esto no significa necesariamente que ordene mal el riesgo. Todavía no hay resultados de test. En la fase 05 revisaremos calibración, curvas ROC/PR, matriz de confusión y validación; cualquier ajuste de umbral o calibración deberá hacerse con entrenamiento/validación y no con test. No existe validación temporal o externa, y el EDA inicial observó todos los registros.